> **Chapter 11, Part 2** | Advanced lens. **Focus:** fractal descriptors as usable features for classification, anomaly detection, and structural comparison.


# Fractal Features for Pattern Recognition

The literature is strongest when fractal descriptors are used as features rather than as slogans. Texture roughness, burstiness, clustering, and graph structure often look different when the observation scale changes. Fractal dimension and lacunarity are two ways of describing that change.

## Outputs

- two synthetic textures with different structural character
- a box-counting estimate for each texture
- a simple lacunarity calculation that captures gap structure

## Supporting reading

- Lopes and Betrouni review: https://pubmed.ncbi.nlm.nih.gov/19535282/
- Gülbay and Kahraman on process-pattern detection: https://www.sciencedirect.com/science/article/abs/pii/S0360835203000925
- Kirichenko et al. on anomalies in fractal time series: https://www.mdpi.com/1099-4300/26/7/581

## Failure note

If two patterns have the same mean intensity and variance, ordinary summary statistics can still miss a structural difference that a scale-sensitive descriptor catches.

## How I would debug this

Render the synthetic patterns first. If you cannot see the structural difference with your eyes, your metrics will be hard to trust.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)


def smooth(grid, rounds=8):
    out = grid.astype(float)
    for _ in range(rounds):
        out = (
            out
            + np.roll(out, 1, 0)
            + np.roll(out, -1, 0)
            + np.roll(out, 1, 1)
            + np.roll(out, -1, 1)
        ) / 5
    return out


def box_count(binary_mask, box_size):
    rows, cols = binary_mask.shape
    count = 0
    for r in range(0, rows, box_size):
        for c in range(0, cols, box_size):
            if np.any(binary_mask[r:r + box_size, c:c + box_size]):
                count += 1
    return count


def estimate_dimension(binary_mask, box_sizes):
    counts = np.array([box_count(binary_mask, s) for s in box_sizes], dtype=float)
    valid = counts > 0
    x = np.log(1 / np.array(box_sizes)[valid])
    y = np.log(counts[valid])
    slope, _ = np.polyfit(x, y, 1)
    return slope


def lacunarity(binary_mask, box_size):
    values = []
    rows, cols = binary_mask.shape
    for r in range(0, rows, box_size):
        for c in range(0, cols, box_size):
            block = binary_mask[r:r + box_size, c:c + box_size]
            values.append(block.sum())
    values = np.array(values, dtype=float)
    return (values.var() / (values.mean() ** 2)) + 1


noise = rng.random((256, 256))
texture_uniform = noise > 0.5
texture_clustered = smooth(rng.random((256, 256)), rounds=12) > 0.52

box_sizes = [2, 4, 8, 16, 32, 64]
results = {
    "uniform": {
        "dimension": estimate_dimension(texture_uniform, box_sizes),
        "lacunarity": lacunarity(texture_uniform, 16),
    },
    "clustered": {
        "dimension": estimate_dimension(texture_clustered, box_sizes),
        "lacunarity": lacunarity(texture_clustered, 16),
    },
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(texture_uniform, cmap="gray")
axes[0].set_title("Texture A: dispersed occupancy")
axes[0].set_axis_off()
axes[1].imshow(texture_clustered, cmap="gray")
axes[1].set_title("Texture B: clustered occupancy")
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

results


## Reading the result

The clustered pattern usually exhibits higher lacunarity because the gaps are more uneven. The box-counting estimate may also shift because occupancy concentrates differently as the observation window changes.

That is the bridge into pattern recognition. These descriptors do not replace domain-specific features. They add structural information that ordinary features often compress away.

## Enterprise translation

The enterprise analogy is not “customer records form a Mandelbrot set.” The better analogy is this: duplicate clusters, hierarchy irregularity, missingness pockets, and lineage fragility may exhibit multi-scale structure. If they do, fractal descriptors become candidates for anomaly detection, triage, or prioritization.

## Exercise

1. change the smoothing rounds and threshold for the clustered texture
2. compute lacunarity at multiple box sizes
3. write down which metric is more sensitive to gap structure in your simulation
